# Evaluation Metrics for Natural Language Tasks

**Master Data & IA – ML Evaluation – 2026**

This notebook demonstrates the evaluation metrics of three NLP tasks with small, reproducible examples:

| # | Task | Metrics | Library |
|---|------|---------|---------|
| 1 | Text Classification | Accuracy, Precision, Recall, F1-score, Confusion Matrix | `scikit-learn` |
| 2 | Named Entity Recognition (NER) | Entity-level Precision, Recall, F1-score | `seqeval` |
| 3 | Machine Translation | BLEU | `sacrebleu` |

No GPU and no pretrained model is required. Run the cells in order (*Runtime → Run all*).

## 0. Setup

In [1]:
# Install the two extra libraries only if they are missing (needed on Colab)
import subprocess, sys
try:
    import seqeval, sacrebleu
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "seqeval", "sacrebleu"])

In [2]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

from seqeval.metrics import (precision_score as ner_precision,
                             recall_score as ner_recall,
                             f1_score as ner_f1,
                             accuracy_score as ner_token_accuracy,
                             classification_report as ner_report)
from seqeval.metrics.sequence_labeling import get_entities
from seqeval.scheme import IOB2

import sacrebleu
from sacrebleu.metrics import BLEU

pd.set_option("display.width", 120)
print("Imports OK")

Imports OK


---
# 1. Text Classification

## Introduction
Text classification assigns one label to a whole text. Here we do **sentiment analysis** with three classes: `negative`, `neutral`, `positive`. We train a very simple model (TF-IDF + Logistic Regression) on a tiny dataset; the goal is to evaluate its predictions, not to build the best model.

## Metrics
Let TP, FP, FN, TN be the true positives, false positives, false negatives and true negatives of a class.

- **Accuracy** = correct predictions / all predictions.
- **Precision** = TP / (TP + FP): of the texts predicted as a class, how many really belong to it.
- **Recall** = TP / (TP + FN): of the texts that truly belong to a class, how many were found.
- **F1-score** = 2·P·R / (P + R): harmonic mean of precision and recall.
- **Confusion matrix**: rows = true class, columns = predicted class; the diagonal holds the correct predictions.
- For several classes, per-class scores are averaged: **macro** (every class counts equally) or **weighted** (classes weighted by their support).

In [3]:
train_data = [
    # negative
    ("The battery died after two days and the support was useless", "negative"),
    ("Terrible quality, the screen cracked on the first week", "negative"),
    ("I hate this phone, it is slow and keeps crashing", "negative"),
    ("Worst purchase ever, a complete waste of money", "negative"),
    ("The delivery was late and the box was damaged", "negative"),
    ("Awful service, nobody answered my emails", "negative"),
    ("The app is buggy and freezes all the time", "negative"),
    ("Very disappointed, the product broke immediately", "negative"),
    ("Poor sound quality and uncomfortable to wear", "negative"),
    ("The food was cold and the waiter was rude", "negative"),
    ("Boring plot and terrible acting, I wanted to leave", "negative"),
    ("It stopped working after a week, totally unacceptable", "negative"),
    # positive
    ("Excellent phone, the battery lasts all day", "positive"),
    ("I love this product, it works perfectly", "positive"),
    ("Great service and very fast delivery", "positive"),
    ("The screen is gorgeous and the camera is amazing", "positive"),
    ("Fantastic quality, I highly recommend it", "positive"),
    ("The staff were friendly and the food was delicious", "positive"),
    ("Wonderful movie with a brilliant cast", "positive"),
    ("Very happy with my purchase, great value for money", "positive"),
    ("The app is smooth, fast and easy to use", "positive"),
    ("Superb sound and very comfortable to wear", "positive"),
    ("Best purchase this year, absolutely perfect", "positive"),
    ("Outstanding support, they solved my problem quickly", "positive"),
    # neutral
    ("The package arrived on Tuesday", "neutral"),
    ("The phone has a six inch screen and 128 GB of storage", "neutral"),
    ("I bought this laptop last month", "neutral"),
    ("The store opens at nine and closes at six", "neutral"),
    ("The movie is two hours long", "neutral"),
    ("The product comes in black and white", "neutral"),
    ("The manual is included in the box", "neutral"),
    ("I ordered the blue model with a charger", "neutral"),
    ("The app was updated to version two", "neutral"),
    ("The restaurant is located near the station", "neutral"),
    ("The headphones have a three meter cable", "neutral"),
    ("The device connects through Bluetooth or USB", "neutral"),
]

test_data = [
    # negative
    ("The screen is terrible and the battery is awful", "negative"),
    ("Rude staff and cold food, very disappointing", "negative"),
    ("The laptop keeps crashing, a waste of money", "negative"),
    ("Useless support and a damaged box", "negative"),
    ("The movie was boring and badly acted", "negative"),
    ("It broke after two days, I hate it", "negative"),
    # positive
    ("Great camera and fantastic battery life", "positive"),
    ("The staff were wonderful and the food was delicious", "positive"),
    ("I love the app, it is fast and easy", "positive"),
    ("Excellent value, I highly recommend this laptop", "positive"),
    ("A brilliant movie with amazing acting", "positive"),
    ("Very happy, the delivery was fast and perfect", "positive"),
    # neutral
    ("The laptop has a 14 inch screen and 16 GB of RAM", "neutral"),
    ("The store is located near the station", "neutral"),
    ("I ordered the black model last week", "neutral"),
    ("The charger is included in the box", "neutral"),
    ("The movie starts at eight", "neutral"),
    ("The phone comes with a USB cable", "neutral"),
]

X_train, y_train = zip(*train_data)
X_test, y_test = zip(*test_data)
print(f"Train: {len(X_train)} texts | Test: {len(X_test)} texts")

Train: 36 texts | Test: 18 texts


In [4]:
# A simple, CPU-only classifier
model = make_pipeline(TfidfVectorizer(), LogisticRegression(C=10, max_iter=1000))
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

results = pd.DataFrame({"text": X_test, "true": y_test, "predicted": y_pred})
results["correct"] = results["true"] == results["predicted"]
results

,text,true,predicted,correct
0,The screen is terrible and the battery is awful,negative,negative,True
1,"Rude staff and cold food, very disappointing",negative,positive,False
2,"The laptop keeps crashing, a waste of money",negative,negative,True
3,Useless support and a damaged box,negative,negative,True
4,The movie was boring and badly acted,negative,negative,True
5,"It broke after two days, I hate it",negative,negative,True
6,Great camera and fantastic battery life,positive,positive,True
7,The staff were wonderful and the food was deli...,positive,positive,True
8,"I love the app, it is fast and easy",positive,positive,True
9,"Excellent value, I highly recommend this laptop",positive,positive,True


### Evaluation

In [5]:
labels = ["negative", "neutral", "positive"]

# Global metrics
print(f"Accuracy          : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision (macro) : {precision_score(y_test, y_pred, average='macro', zero_division=0):.3f}")
print(f"Recall    (macro) : {recall_score(y_test, y_pred, average='macro', zero_division=0):.3f}")
print(f"F1-score  (macro) : {f1_score(y_test, y_pred, average='macro', zero_division=0):.3f}")

# Per-class metrics
print("\nPer-class report:")
print(classification_report(y_test, y_pred, labels=labels, digits=3, zero_division=0))

Accuracy          : 0.944
Precision (macro) : 0.952
Recall    (macro) : 0.944
F1-score  (macro) : 0.944

Per-class report:
              precision    recall  f1-score   support

    negative      1.000     0.833     0.909         6
     neutral      1.000     1.000     1.000         6
    positive      0.857     1.000     0.923         6

    accuracy                          0.944        18
   macro avg      0.952     0.944     0.944        18
weighted avg      0.952     0.944     0.944        18



In [6]:
# Confusion matrix (rows = true class, columns = predicted class)
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm,
                     index=[f"true_{l}" for l in labels],
                     columns=[f"pred_{l}" for l in labels])
cm_df

,pred_negative,pred_neutral,pred_positive
true_negative,5,0,1
true_neutral,0,6,0
true_positive,0,0,6


### Check: metrics recomputed by hand from the confusion matrix

In [7]:
for i, label in enumerate(labels):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    print(f"{label:9s} TP={tp} FP={fp} FN={fn} -> P={p:.3f} R={r:.3f} F1={f:.3f}")

manual_acc = np.trace(cm) / cm.sum()
assert abs(manual_acc - accuracy_score(y_test, y_pred)) < 1e-12
print(f"\nManual accuracy = trace / total = {manual_acc:.3f}  (matches scikit-learn)")

negative  TP=5 FP=0 FN=1 -> P=1.000 R=0.833 F1=0.909
neutral   TP=6 FP=0 FN=0 -> P=1.000 R=1.000 F1=1.000
positive  TP=6 FP=1 FN=0 -> P=0.857 R=1.000 F1=0.923

Manual accuracy = trace / total = 0.944  (matches scikit-learn)


---
# 2. Named Entity Recognition (NER)

## Introduction
NER finds and types entity mentions in text (e.g. `PER`, `ORG`, `LOC`). It is a **sequence labeling** task: each token gets a tag in the **BIO** scheme (`B-` = beginning of an entity, `I-` = inside, `O` = outside).

## Metrics
Token-level accuracy is misleading for NER because most tokens are `O`. NER is evaluated at the **entity level**: a predicted entity is correct only if its **type and its exact boundaries** (start and end) both match a gold entity.

- **Precision** = correct predicted entities / all predicted entities
- **Recall** = correct predicted entities / all gold entities
- **F1-score** = harmonic mean of precision and recall

`seqeval` implements this entity-level evaluation. Scores can be reported per entity type, or aggregated as **micro** (pool all entities), **macro**, or **weighted** averages.

## Example
Five hand-annotated sentences with gold tags (`y_true`) and the output of a hypothetical NER system (`y_pred`), containing typical errors: a boundary error, a type error, a missed entity and a spurious entity.

In [8]:
sentences = [
    "Elon Musk founded SpaceX in California .",
    "Angela Merkel met Emmanuel Macron in Berlin .",
    "Google opened an office in New York .",
    "Marie Curie worked at the University of Paris .",
    "Microsoft hired John Smith in Casablanca .",
]

y_true = [
    ["B-PER", "I-PER", "O", "B-ORG", "O", "B-LOC", "O"],
    ["B-PER", "I-PER", "O", "B-PER", "I-PER", "O", "B-LOC", "O"],
    ["B-ORG", "O", "O", "O", "O", "B-LOC", "I-LOC", "O"],
    ["B-PER", "I-PER", "O", "O", "O", "B-ORG", "I-ORG", "I-ORG", "O"],
    ["B-ORG", "O", "B-PER", "I-PER", "O", "B-LOC", "O"],
]

y_pred = [
    ["B-PER", "I-PER", "O", "B-ORG", "O", "B-LOC", "O"],              # all correct
    ["B-PER", "I-PER", "O", "B-PER", "O", "O", "B-LOC", "O"],         # boundary error: "Emmanuel" only
    ["B-ORG", "O", "O", "O", "O", "B-ORG", "I-ORG", "O"],             # type error: "New York" tagged ORG
    ["B-PER", "I-PER", "O", "O", "O", "O", "O", "B-LOC", "O"],        # missed ORG + spurious LOC "Paris"
    ["B-ORG", "O", "B-PER", "I-PER", "O", "B-LOC", "O"],              # all correct
]

# Sanity check: same number of tags as tokens
for s, t, p in zip(sentences, y_true, y_pred):
    assert len(s.split()) == len(t) == len(p), s

# Show the entities extracted from the tags: (type, start, end)
for s, t, p in zip(sentences, y_true, y_pred):
    print(s)
    print("  gold:", get_entities(t))
    print("  pred:", get_entities(p))

Elon Musk founded SpaceX in California .
  gold: [('PER', 0, 1), ('ORG', 3, 3), ('LOC', 5, 5)]
  pred: [('PER', 0, 1), ('ORG', 3, 3), ('LOC', 5, 5)]
Angela Merkel met Emmanuel Macron in Berlin .
  gold: [('PER', 0, 1), ('PER', 3, 4), ('LOC', 6, 6)]
  pred: [('PER', 0, 1), ('PER', 3, 3), ('LOC', 6, 6)]
Google opened an office in New York .
  gold: [('ORG', 0, 0), ('LOC', 5, 6)]
  pred: [('ORG', 0, 0), ('ORG', 5, 6)]
Marie Curie worked at the University of Paris .
  gold: [('PER', 0, 1), ('ORG', 5, 7)]
  pred: [('PER', 0, 1), ('LOC', 7, 7)]
Microsoft hired John Smith in Casablanca .
  gold: [('ORG', 0, 0), ('PER', 2, 3), ('LOC', 5, 5)]
  pred: [('ORG', 0, 0), ('PER', 2, 3), ('LOC', 5, 5)]


### Evaluation

In [9]:
print(f"Entity-level Precision : {ner_precision(y_true, y_pred):.3f}")
print(f"Entity-level Recall    : {ner_recall(y_true, y_pred):.3f}")
print(f"Entity-level F1-score  : {ner_f1(y_true, y_pred):.3f}")
print(f"(Token-level accuracy  : {ner_token_accuracy(y_true, y_pred):.3f}  <- misleading, inflated by 'O' tokens)")

print("\nPer-entity-type report (strict IOB2 mode):")
print(ner_report(y_true, y_pred, mode="strict", scheme=IOB2, digits=3))

Entity-level Precision : 0.769
Entity-level Recall    : 0.769
Entity-level F1-score  : 0.769
(Token-level accuracy  : 0.846  <- misleading, inflated by 'O' tokens)

Per-entity-type report (strict IOB2 mode):
              precision    recall  f1-score   support

         LOC      0.750     0.750     0.750         4
         ORG      0.750     0.750     0.750         4
         PER      0.800     0.800     0.800         5

   micro avg      0.769     0.769     0.769        13
   macro avg      0.767     0.767     0.767        13
weighted avg      0.769     0.769     0.769        13



### Check: TP / FP / FN counted by hand

In [10]:
tp = fp = fn = 0
for t, p in zip(y_true, y_pred):
    gold, pred = set(get_entities(t)), set(get_entities(p))
    tp += len(gold & pred)      # exact match: same type, start and end
    fp += len(pred - gold)      # predicted but not in gold
    fn += len(gold - pred)      # in gold but not predicted

precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
print(f"TP={tp} FP={fp} FN={fn}")
print(f"Precision={precision:.3f} Recall={recall:.3f} F1={f1:.3f}")

assert abs(precision - ner_precision(y_true, y_pred)) < 1e-12
assert abs(recall - ner_recall(y_true, y_pred)) < 1e-12
assert abs(f1 - ner_f1(y_true, y_pred)) < 1e-12
print("Manual computation matches seqeval.")

TP=10 FP=3 FN=3
Precision=0.769 Recall=0.769 F1=0.769
Manual computation matches seqeval.


---
# 3. Machine Translation

## Introduction
Machine translation (MT) automatically translates text from a source language to a target language. A system's output (the *hypothesis*) is compared with one or more human translations (the *references*). Here we evaluate English translations of French sentences. The hypotheses are written by hand to simulate the output of two MT systems, A and B.

## Metric: BLEU
**BLEU** (Bilingual Evaluation Understudy) measures the overlap of n-grams between the hypothesis and the reference(s):

1. **Modified n-gram precision** $p_n$ for n = 1…4: the fraction of hypothesis n-grams that also appear in the reference (each reference n-gram can be matched only as many times as it occurs there).
2. **Brevity penalty** $BP = \min(1, e^{1 - r/c})$, where $c$ is the hypothesis length and $r$ the reference length. It penalizes translations that are too short.
3. $BLEU = BP \cdot \exp\left(\frac{1}{4}\sum_{n=1}^{4}\log p_n\right)$, reported on a 0–100 scale.

Scores are computed at the **corpus level** (n-gram statistics are accumulated over all sentences) with `sacrebleu`, the standard implementation that ensures reproducible, comparable scores. Sentence-level BLEU is noisy and mostly useful for inspection.

In [11]:
sources = [   # French source sentences (for information only, BLEU does not use them)
    "Le chat est assis sur le tapis.",
    "Je voudrais réserver une table pour deux personnes ce soir.",
    "Le temps à Rabat est très agréable aujourd'hui.",
    "Elle étudie l'apprentissage automatique depuis trois ans.",
    "Où est la gare la plus proche ?",
]

references = [
    "The cat is sitting on the mat.",
    "I would like to book a table for two people tonight.",
    "The weather in Rabat is very nice today.",
    "She has been studying machine learning for three years.",
    "Where is the nearest train station?",
]

system_A = [   # good system: mostly correct, a few wording differences
    "The cat is sitting on the mat.",
    "I would like to reserve a table for two people tonight.",
    "The weather in Rabat is very good today.",
    "She studies machine learning since three years.",
    "Where is the nearest station of train?",
]

system_B = [   # weaker system: wrong words and truncated outputs
    "The cat sits on carpet.",
    "I want a table two persons this evening.",
    "The time in Rabat is pleasant.",
    "She studies learning automatic for three years.",
    "Where is the station?",
]

pd.DataFrame({"source (FR)": sources, "reference": references,
              "system A": system_A, "system B": system_B})

,source (FR),reference,system A,system B
0,Le chat est assis sur le tapis.,The cat is sitting on the mat.,The cat is sitting on the mat.,The cat sits on carpet.
1,Je voudrais réserver une table pour deux perso...,I would like to book a table for two people to...,I would like to reserve a table for two people...,I want a table two persons this evening.
2,Le temps à Rabat est très agréable aujourd'hui.,The weather in Rabat is very nice today.,The weather in Rabat is very good today.,The time in Rabat is pleasant.
3,Elle étudie l'apprentissage automatique depuis...,She has been studying machine learning for thr...,She studies machine learning since three years.,She studies learning automatic for three years.
4,Où est la gare la plus proche ?,Where is the nearest train station?,Where is the nearest station of train?,Where is the station?


### Evaluation

In [12]:
# sacrebleu expects: hypotheses = list of strings, references = list of reference *streams*
bleu_A = sacrebleu.corpus_bleu(system_A, [references])
bleu_B = sacrebleu.corpus_bleu(system_B, [references])

print(f"System A  BLEU = {bleu_A.score:.2f}")
print(f"System B  BLEU = {bleu_B.score:.2f}")
print()
print(bleu_A)   # includes n-gram precisions, brevity penalty and lengths
print(bleu_B)

System A  BLEU = 62.42
System B  BLEU = 15.26

BLEU = 62.42 88.9/70.0/57.1/46.7 (BP = 0.978 ratio = 0.978 hyp_len = 45 ref_len = 46)
BLEU = 15.26 71.4/33.3/16.0/5.0 (BP = 0.730 ratio = 0.761 hyp_len = 35 ref_len = 46)


In [13]:
# Components of the score
rows = []
for name, b in [("System A", bleu_A), ("System B", bleu_B)]:
    rows.append({"system": name, "BLEU": round(b.score, 2),
                 "p1": round(b.precisions[0], 1), "p2": round(b.precisions[1], 1),
                 "p3": round(b.precisions[2], 1), "p4": round(b.precisions[3], 1),
                 "BP": round(b.bp, 3), "hyp_len": b.sys_len, "ref_len": b.ref_len})
pd.DataFrame(rows).set_index("system")

,BLEU,p1,p2,p3,p4,BP,hyp_len,ref_len
system,,,,,,,,
System A,62.42,88.9,70.0,57.1,46.7,0.978,45,46
System B,15.26,71.4,33.3,16.0,5.0,0.730,35,46


In [14]:
# Sentence-level BLEU (useful to see which sentences hurt the score)
sent_rows = []
for i, ref in enumerate(references):
    sent_rows.append({
        "reference": ref,
        "BLEU_A": round(sacrebleu.sentence_bleu(system_A[i], [ref]).score, 1),
        "BLEU_B": round(sacrebleu.sentence_bleu(system_B[i], [ref]).score, 1),
    })
pd.DataFrame(sent_rows)

,reference,BLEU_A,BLEU_B
0,The cat is sitting on the mat.,100.0,13.8
1,I would like to book a table for two people to...,73.5,8.6
2,The weather in Rabat is very nice today.,66.1,20.9
3,She has been studying machine learning for thr...,21.1,29.8
4,Where is the nearest train station?,39.8,33.5


### Effect of the brevity penalty and of multiple references

In [15]:
ref = ["The cat is sitting on the mat."]

# 1) A hypothesis that is correct but too short: all its n-grams match, yet BP < 1
short = sacrebleu.corpus_bleu(["The cat is sitting on"], [ref])
full = sacrebleu.corpus_bleu(["The cat is sitting on the mat."], [ref])
print(f"Too short : BLEU={short.score:6.2f}  BP={short.bp:.3f}  (hyp_len={short.sys_len}, ref_len={short.ref_len})")
print(f"Exact     : BLEU={full.score:6.2f}  BP={full.bp:.3f}")

# 2) Several valid references: the hypothesis is matched against all of them
hyp = ["I would like to reserve a table for two people tonight."]
one_ref = [["I would like to book a table for two people tonight."]]
two_refs = [["I would like to book a table for two people tonight."],
            ["I would like to reserve a table for two people tonight."]]
print(f"\nOne reference  : BLEU={sacrebleu.corpus_bleu(hyp, one_ref).score:.2f}")
print(f"Two references : BLEU={sacrebleu.corpus_bleu(hyp, two_refs).score:.2f}")

Too short : BLEU= 54.88  BP=0.549  (hyp_len=5, ref_len=8)
Exact     : BLEU=100.00  BP=1.000

One reference  : BLEU=73.49
Two references : BLEU=100.00
